<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/03_Baseline_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")
import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

df = load_dataset(PROJECT_PATH)

X, y, groups = split_xy(df)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Class distribution:")
print(y.value_counts())


Mounted at /content/drive
Dataset shape: (756, 753)
Target shape: (756,)
Class distribution:
class
1    564
0    192
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


Training shape: (603, 753)
Testing shape: (153, 753)


In [3]:
duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

print("Number of duplicate features:", len(duplicate_columns))

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

print("Training shape after duplicate removal:", X_train.shape)
print("Testing shape after duplicate removal:", X_test.shape)


Number of duplicate features: 0
Constant features removed: 35
Training shape after duplicate removal: (603, 718)
Testing shape after duplicate removal: (153, 718)


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="linear",
            class_weight="balanced"
        ))
    ]),

    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            class_weight="balanced"
        ))
    ])
}

print("Baseline models defined successfully.")

Baseline models defined successfully.


In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = make_cv(groups_train)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro"
}

cv_results = {}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    cv_results[name] = {
        "Accuracy": scores["test_accuracy"].mean(),
        "Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
        "F1 Macro": scores["test_f1_macro"].mean()
    }

results_df = pd.DataFrame(cv_results).T

print(results_df.round(4))


                     Accuracy  Balanced Accuracy  F1 Macro
Logistic Regression    0.7765             0.7271    0.7157
Linear SVM             0.7665             0.7067    0.6985
RBF SVM                0.7778             0.7101    0.7042


In [6]:
best_model_name = results_df["F1 Macro"].idxmax()

print("Best baseline model based on CV F1 Macro:")
print(best_model_name)

print("\nBest model results:")
print(results_df.loc[best_model_name])


Best baseline model based on CV F1 Macro:
Logistic Regression

Best model results:
Accuracy             0.776463
Balanced Accuracy    0.727050
F1 Macro             0.715697
Name: Logistic Regression, dtype: float64


In [7]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

test_results = {}

fitted_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    fitted_models[name] = model

    test_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision Macro": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall Macro": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "F1 Macro": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    }

test_results_df = pd.DataFrame(test_results).T

print(test_results_df.round(4))

                     Accuracy  Balanced Accuracy  Precision Macro  \
Logistic Regression    0.7647             0.7240           0.6999   
Linear SVM             0.7843             0.7287           0.7181   
RBF SVM                0.7712             0.7031           0.6998   

                     Recall Macro  F1 Macro  
Logistic Regression        0.7240    0.7089  
Linear SVM                 0.7287    0.7230  
RBF SVM                    0.7031    0.7014  


In [8]:
best_model = fitted_models[best_model_name]

y_pred_best = best_model.predict(X_test)

print("Best model:", best_model_name)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))


Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.64      0.58        39
           1       0.87      0.81      0.84       114

    accuracy                           0.76       153
   macro avg       0.70      0.72      0.71       153
weighted avg       0.78      0.76      0.77       153


Confusion Matrix:
[[25 14]
 [22 92]]


In [10]:
RESULTS_PATH = get_results_path(PROJECT_PATH)

results_df.to_csv(
    RESULTS_PATH / "baseline_cv_results.csv"
)

test_results_df.to_csv(
    RESULTS_PATH / "baseline_test_results.csv"
)

print("Baseline results saved successfully")


Baseline results saved successfully
